In [0]:
from pyspark.sql.functions import col, when, count, avg

df = spark.table("workspace.default.silver_hospital_clean")

df_gold = df.withColumn("high_medication_flag",
        when(col("num_medications") > 15, 1).otherwise(0)) \
    .withColumn("high_procedures_flag",
        when(col("num_procedures") > 3, 1).otherwise(0)) \
    .withColumn("frequent_visitor_flag",
        when((col("number_outpatient") + col("number_emergency") + 
              col("number_inpatient")) > 3, 1).otherwise(0)) \
    .withColumn("long_stay_flag",
        when(col("time_in_hospital") > 7, 1).otherwise(0)) \
    .withColumn("insulin_flag",
        when(col("insulin").isin(["Steady","Up","Down"]), 1).otherwise(0)) \
    .withColumn("diabetes_primary_flag",
        when(col("diag_1").startswith("250"), 1).otherwise(0))

print(f"Gold rows: {df_gold.count()}")
df_gold.select("high_medication_flag","high_procedures_flag",
               "frequent_visitor_flag","long_stay_flag",
               "insulin_flag","diabetes_primary_flag",
               "readmitted_binary").show(5)

Gold rows: 97108
+--------------------+--------------------+---------------------+--------------+------------+---------------------+-----------------+
|high_medication_flag|high_procedures_flag|frequent_visitor_flag|long_stay_flag|insulin_flag|diabetes_primary_flag|readmitted_binary|
+--------------------+--------------------+---------------------+--------------+------------+---------------------+-----------------+
|                   1|                   1|                    0|             0|           1|                    0|                1|
|                   0|                   0|                    0|             0|           1|                    0|                1|
|                   1|                   0|                    0|             1|           1|                    0|                1|
|                   0|                   0|                    0|             0|           1|                    0|                1|
|                   1|                   0|  

In [0]:
# Department level aggregations
dept_summary = df_gold.groupBy("admission_type_id").agg(
    count("encounter_id").alias("total_patients"),
    avg("time_in_hospital").alias("avg_length_of_stay"),
    avg("num_medications").alias("avg_medications"),
    avg("readmitted_binary").alias("readmission_rate")
).orderBy("admission_type_id")

dept_summary.display()

admission_type_id,total_patients,avg_length_of_stay,avg_medications,readmission_rate
1,51306,4.3625112072662064,15.323958211515222,0.488519861224808
2,17445,4.6047578102608195,15.035998853539697,0.47818859271997705
3,18311,4.307410845939599,18.596799737862487,0.4149418382393097
4,10,3.2,11.6,0.3
5,4561,3.912738434553826,15.901775926331945,0.48169261126945845
6,5141,4.56876094145108,16.46177786422875,0.5399727679439797
7,17,5.470588235294118,17.11764705882353,0.0
8,317,3.082018927444795,17.50473186119874,0.3470031545741325


In [0]:
#  Save Gold Delta table:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_hospital_features")

dept_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_dept_summary")

print("Gold tables saved successfully")

Gold tables saved successfully
